# Full-panel host–phage association analysis
### 16S rRNA phylogeny + ESM2/RBH prophage relatedness -> Mantel test

This notebook constructs the broader 16S rRNA host phylogeny and evaluates the association between host phylogenetic distance and prophage-relatedness distance for the 17 genomes carrying at least one geNomad-confirmed prophage.

The result is a comparative genomic association and should not be interpreted as direct evidence of contemporary phage transmission or circulation.


## 1. Install tools

In [ ]:
!apt-get install -y mafft fasttree -q 2>&1 | tail -5
!pip install -q dendropy scikit-bio biopython


## 2. Upload Final_Genomes.zip and unzip

In [ ]:
from google.colab import files
import os, zipfile

UPLOAD_DIR = "/content/uploaded"
os.makedirs(UPLOAD_DIR, exist_ok=True)
print("Select Final_Genomes.zip:")
uploaded = files.upload()
for fname in uploaded:
    if os.path.exists(fname):
        os.rename(fname, f"{UPLOAD_DIR}/{fname}")


In [ ]:
UNZIP_DIR = "/content/inputs"
os.makedirs(UNZIP_DIR, exist_ok=True)
zpath = f"{UPLOAD_DIR}/Final_Genomes.zip"
target = f"{UNZIP_DIR}/Final_Genomes"
if not os.path.isdir(target) or not os.listdir(target):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(target)

cur = target
while True:
    entries = [e for e in os.listdir(cur) if not e.startswith("__MACOSX")]
    subdirs = [e for e in entries if os.path.isdir(os.path.join(cur, e))]
    if len(subdirs) == 1 and not any(e.startswith(("GCA_", "GCF_")) for e in entries):
        cur = os.path.join(cur, subdirs[0])
    else:
        break
GENOMES_DIR = cur

accessions = sorted(d for d in os.listdir(GENOMES_DIR)
                     if os.path.isdir(os.path.join(GENOMES_DIR, d)) and d.startswith(("GCA_", "GCF_"))
                     and os.path.exists(os.path.join(GENOMES_DIR, d, "genomic.gbff")))
print(f"Found {len(accessions)} genome accessions")


## 3. Extract 16S rRNA sequences

In [ ]:
from Bio import SeqIO

records_16s = {}
for acc in accessions:
    gbff = os.path.join(GENOMES_DIR, acc, "genomic.gbff")
    candidates = []
    for rec in SeqIO.parse(gbff, "genbank"):
        for feat in rec.features:
            if feat.type == "rRNA":
                product = str(feat.qualifiers.get("product", [""])[0]).lower()
                if "16s" in product:
                    candidates.append(str(feat.extract(rec.seq)))
    if candidates:
        records_16s[acc] = max(candidates, key=len)  # longest copy of multiple near-identical operons
        print(f"{acc}: {len(candidates)} copies, using longest ({len(records_16s[acc])} bp)")
    else:
        print(f"{acc}: NO 16S rRNA found -- will be excluded from the tree")

print(f"\nTotal genomes with 16S extracted: {len(records_16s)} / {len(accessions)}")
with open("/content/16S_sequences.fasta", "w") as f:
    for acc, seq in records_16s.items():
        f.write(f">{acc}\n{seq}\n")


## 4. Align (MAFFT) and build tree (FastTree, GTR model)

In [ ]:
!mafft --auto /content/16S_sequences.fasta > /content/16S_aligned.fasta 2>/content/mafft.log
!fasttree -nt -gtr /content/16S_aligned.fasta > /content/16S_tree.nwk 2>/content/fasttree.log
!tail -5 /content/fasttree.log
print()
with open("/content/16S_tree.nwk") as f:
    print(f.read())


## 5. Load master table + embeddings (upload if not already in memory)

In [ ]:
import pandas as pd, numpy as np

def ensure_file(filename):
    if os.path.exists(filename):
        return filename
    print(f"Please upload {filename}:")
    up = files.upload()
    for f in up:
        if f != filename:
            os.rename(f, filename)
    return filename

master_path = ensure_file("MASTER_prophage_validation_table.csv")
master = pd.read_csv(master_path)

if "embeddings" in globals() and len(globals()["embeddings"]) > 0:
    print(f"Using in-memory embeddings ({len(embeddings)} proteins)")
else:
    emb_path = ensure_file("esm2_embeddings.npz")
    embeddings = dict(np.load(emb_path, allow_pickle=True))
    print(f"Loaded {len(embeddings)} embeddings")


## 6. Anisotropy-corrected prophage distance (same method as the network/earlier Mantel test)

In [ ]:
from numpy.linalg import norm

all_vecs = np.vstack(list(embeddings.values()))
global_mean = all_vecs.mean(axis=0)
centered = {k: v - global_mean for k, v in embeddings.items()}

confirmed_headers = master[master["genomad_call"] == "virus"]["header"].tolist()
by_prophage = {h: [k for k in embeddings if k.startswith(h + "__")] for h in confirmed_headers}

def rbh_sim(h1, h2):
    p1, p2 = by_prophage[h1], by_prophage[h2]
    V1 = np.vstack([centered[p] for p in p1]); V1 = V1 / (norm(V1, axis=1, keepdims=True) + 1e-9)
    V2 = np.vstack([centered[p] for p in p2]); V2 = V2 / (norm(V2, axis=1, keepdims=True) + 1e-9)
    S = V1 @ V2.T
    return float(np.concatenate([S.max(axis=1), S.max(axis=0)]).mean())

print(f"{len(confirmed_headers)} geNomad-confirmed prophages available for comparison")


## 7. Build host + phage distance matrices, run Mantel test at full power

In [ ]:
import dendropy
from skbio.stats.distance import mantel, DistanceMatrix

tree = dendropy.Tree.get(path="/content/16S_tree.nwk", schema="newick", preserve_underscores=True)
pdm = tree.phylogenetic_distance_matrix()
taxon_by_label = {t.label: t for t in tree.taxon_namespace}
print("Tree tips:", len(taxon_by_label))

acc_of = master.set_index("header")["accession"].to_dict()
usable_accs = sorted(set(a for a in taxon_by_label if any(acc_of.get(h) == a for h in confirmed_headers)))
acc_to_headers = {a: [h for h in confirmed_headers if acc_of.get(h) == a] for a in usable_accs}
print(f"Usable genomes (tree tip + >=1 confirmed prophage): {len(usable_accs)} / {len(taxon_by_label)}")

n = len(usable_accs)
host_arr = np.zeros((n, n))
phage_arr = np.zeros((n, n))
for i, a in enumerate(usable_accs):
    for j, b in enumerate(usable_accs):
        host_arr[i, j] = pdm.patristic_distance(taxon_by_label[a], taxon_by_label[b])
        if i == j:
            phage_arr[i, j] = 0.0
        elif i < j:
            ha, hb = acc_to_headers[a], acc_to_headers[b]
            dists = [1 - rbh_sim(h1, h2) for h1 in ha for h2 in hb]
            phage_arr[i, j] = phage_arr[j, i] = float(np.mean(dists))

host_arr = (host_arr + host_arr.T) / 2
np.fill_diagonal(host_arr, 0.0)

host_dm = DistanceMatrix(host_arr, ids=usable_accs)
phage_dm = DistanceMatrix(phage_arr, ids=usable_accs)

r, p, nn = mantel(host_dm, phage_dm, method="pearson", permutations=999)
r_s, p_s, nn_s = mantel(host_dm, phage_dm, method="spearman", permutations=999)

print(f"\n20-genome (16S-based) Mantel test:")
print(f"  N genomes: {nn}, N pairs: {nn*(nn-1)//2}")
print(f"  Pearson  r = {r:.3f}, p = {p:.4f}")
print(f"  Spearman r = {r_s:.3f}, p = {p_s:.4f}")
print(f"\n  Reference (verified run): Pearson r=0.470 p=0.005, Spearman r=0.427 p=0.022")
print(f"  Small differences (~0.01-0.03 in p) are expected -- Mantel p-values come from random permutation.")

pd.DataFrame(host_arr, index=usable_accs, columns=usable_accs).to_csv("/content/host_dist_20genome.csv")
pd.DataFrame(phage_arr, index=usable_accs, columns=usable_accs).to_csv("/content/phage_dist_20genome.csv")


## 8. Robustness check: leave-one-out

In [ ]:
host_df = pd.DataFrame(host_arr, index=usable_accs, columns=usable_accs)
phage_df = pd.DataFrame(phage_arr, index=usable_accs, columns=usable_accs)

print("Leave-one-out robustness (Pearson r, p) -- should stay significant for every genome dropped:")
for drop in usable_accs:
    keep = [a for a in usable_accs if a != drop]
    h = DistanceMatrix(host_df.loc[keep, keep].values, ids=keep)
    ph = DistanceMatrix(phage_df.loc[keep, keep].values, ids=keep)
    r, p, n = mantel(h, ph, method="pearson", permutations=999)
    print(f"  drop {drop:20s}  r={r:.3f}  p={p:.4f}")


## 9. Figure: host distance vs. phage distance, colored by same/different species

In [ ]:
import matplotlib.pyplot as plt

species_map = {
    'GCA_000392485.2':'L. plantarum','GCF_000008065.1':'L. johnsonii','GCF_000010145.1':'L. fermentum',
    'GCF_000011985.1':'L. acidophilus','GCF_000014425.1':'L. gasseri','GCF_000014525.1':'L. paracasei',
    'GCF_000016825.1':'L. reuteri','GCF_000019245.4':'L. paracasei','GCF_000023085.1':'L. plantarum',
    'GCF_000026505.1':'L. rhamnosus','GCF_000148815.2':'L. plantarum','GCF_000165775.1':'L. helveticus',
    'GCF_000203855.3':'L. plantarum','GCF_000248095.2':'L. mucosae','GCF_000338115.2':'L. plantarum',
    'GCF_000389675.2':'L. acidophilus','GCF_000412205.1':'L. plantarum','GCF_001704335.1':'L. plantarum',
    'GCF_014131735.1':'L. plantarum','GCF_041888805.1':'L. reuteri',
}

pairs_h, pairs_p, same_sp = [], [], []
for i in range(n):
    for j in range(i+1, n):
        a, b = usable_accs[i], usable_accs[j]
        pairs_h.append(host_arr[i, j]); pairs_p.append(phage_arr[i, j])
        same_sp.append(species_map.get(a) == species_map.get(b))
pairs_h, pairs_p, same_sp = np.array(pairs_h), np.array(pairs_p), np.array(same_sp)

plt.figure(figsize=(7, 5.5))
plt.scatter(pairs_h[same_sp], pairs_p[same_sp], c="#2b6cb0", label="same species", alpha=0.7, s=40)
plt.scatter(pairs_h[~same_sp], pairs_p[~same_sp], c="#c05621", label="different species", alpha=0.7, s=40)
z = np.polyfit(pairs_h, pairs_p, 1)
xline = np.linspace(pairs_h.min(), pairs_h.max(), 50)
plt.plot(xline, np.polyval(z, xline), "k--", alpha=0.5, label="linear trend")
plt.xlabel("Host phylogenetic distance (16S patristic)")
plt.ylabel("Prophage embedding distance")
plt.title(f"Cophylogeny signal across {nn} genomes\nPearson r={r:.3f}, p={p:.4f}  |  Spearman r={r_s:.3f}, p={p_s:.4f}")
plt.legend()
plt.tight_layout()
plt.savefig("/content/Figure_cophylogeny_20genome.png", dpi=200)
plt.show()


## 10. Download everything

Same lesson as before: download now, before closing the tab.


In [ ]:
from google.colab import files as colab_files
import shutil

shutil.make_archive("/content/cophylogeny_20genome_results", "zip", "/content",
                     base_dir=None)  # fallback if selective copy below fails
import os
bundle_dir = "/content/cophylo_bundle"
os.makedirs(bundle_dir, exist_ok=True)
for fname in ["16S_tree.nwk", "16S_aligned.fasta", "host_dist_20genome.csv",
              "phage_dist_20genome.csv", "Figure_cophylogeny_20genome.png"]:
    src = f"/content/{fname}"
    if os.path.exists(src):
        shutil.copy(src, bundle_dir)

zip_path = "/content/cophylogeny_20genome_results"
shutil.make_archive(zip_path, "zip", bundle_dir)
colab_files.download(f"{zip_path}.zip")
